# This notebook is used to build a linear regression model using the medical insurance dataset

## These are some of the prerequisites to run this notebook when running in the community ediction of databricks

1. Upload the medinsurance_linearreg_class1_csv dataset file from your system and copy that path since that will be used in the cell where we read data

## Setup PySpark in Google Colab

In [4]:
# Install Java Development Kit (JDK)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [5]:
# Install PySpark
!pip install pyspark==3.3.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.3/281.3 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 9.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.3.0-py2.py3-none-any.whl size=281764007 sha256=00e69e517ddfd5ec43ae84da68e9af8416e18d3ef4a891c452bfddc67461d72c
  Stored in directory: /root/.cache/pip/wheels/fa/38/b6/02a8fa3ddb890ded2545483c78766bb378d8f505dc4f26d917
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conne

In [6]:
# Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"

In [7]:
# Initialize SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .appName("MedicalInsuranceRegression") \
    .getOrCreate()
print("SparkSession initialized.")

SparkSession initialized.


In [9]:
# Data processing
from pyspark.sql.functions import log, col, exp

# Modeling
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


In [10]:
medins_chrgs = spark.read.csv('/content/sample_data/medinsurance_linearreg_class1.csv', header=True, inferSchema=True)

#Show basic summary stats
display(medins_chrgs.summary())

DataFrame[summary: string, age: string, bmi: string, children: string, region: string, charges: string, gender_cd: string, smoker_cd: string]

In [11]:
#One hot encoding of string feature Region
medins_chrgs.groupBy('region').count().show()

+---------+-----+
|   region|count|
+---------+-----+
|northwest|  325|
|southeast|  364|
|northeast|  324|
|southwest|  325|
+---------+-----+



In [12]:
# Train test split
trainDF, testDF = medins_chrgs.randomSplit([.65, .35], seed=123)
# Print the number of records
print(f'There are {trainDF.cache().count()} records in the training dataset.')
print(f'There are {testDF.cache().count()} records in the testing dataset.')

There are 872 records in the training dataset.
There are 466 records in the testing dataset.


In [14]:
#One hot encoding of string feature Region
trainDF.groupBy('region').count().show()

+---------+-----+
|   region|count|
+---------+-----+
|northwest|  230|
|southeast|  233|
|northeast|  202|
|southwest|  207|
+---------+-----+



##Now we need to modify the categorical variable region into one-hot-encoded version
 For this we can either use the StringIndexer and OneHotEncoder separately OR use a pipeline to do this in one step

Some machine learning algorithms, such as linear and logistic regression, require numeric features.  

The following code block illustrates how to use `StringIndexer` and `OneHotEncoder` to convert categorical variables into a set of numeric variables that only take on values 0 and 1.

- `StringIndexer` converts a column of string values to a column of label indexes. For example, it might convert the values "red", "blue", and "green" to 0, 1, and 2.
- `OneHotEncoder` maps a column of category indices to a column of binary vectors, with at most one "1" in each row that indicates the category index for that row.

One-hot encoding in Spark is a two-step process. You first use the StringIndexer, followed by the OneHotEncoder. The following code block defines the StringIndexer and OneHotEncoder but does not apply it to any data yet.

For more information:   
[StringIndexer](http://spark.apache.org/docs/latest/ml-features.html#stringindexer)   
[OneHotEncoder](https://spark.apache.org/docs/latest/ml-features.html#onehotencoder)


In [15]:
# #import required libraries
# from pyspark.ml.feature import StringIndexer
# qualification_indexer = StringIndexer(inputCol="region", outputCol="regionIndex")

# #Fits a model to the input dataset with optional parameters.
# trainDF1 = qualification_indexer.fit(trainDF).transform(trainDF)
# trainDF1.show()

In [16]:
# from pyspark.ml.feature import OneHotEncoder
# #onehotencoder to region
# onehotencoder_region_vector = OneHotEncoder(inputCol="regionIndex", outputCol="region_vec")
# trainDF11 = onehotencoder_region_vector.fit(trainDF1).transform(trainDF1)
# trainDF11.show()

## Note: Transformers, Estimators, and Pipelines

Three important concepts in MLlib machine learning that will be used in this notebook, and most others are **Transformers**, **Estimators**, and **Pipelines**.

- **Transformer**: Takes a DataFrame as input, and returns a new DataFrame. Transformers do not learn any parameters from the data and simply apply rule-based transformations to either prepare data for model training or generate predictions using a trained MLlib model. You call a transformer with a `.transform()` method.

- **Estimator**: Learns (or "fits") parameters from your DataFrame via a `.fit()` method and returns a Model, which is a transformer.

- **Pipeline**: Combines multiple steps into a single workflow that can be easily run. Creating a machine learning model typically involves setting up many different steps and iterating over them. Pipelines help you automate this process.

For more information:
[ML Pipelines](https://spark.apache.org/docs/latest/ml-pipeline.html#ml-pipelines)

In [17]:
#You can also create a pipeline and do everything together in one easy fit and transform step
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

categoricalColumns = ["region"]
stages = [] # stages in Pipeline
for categoricalCol in categoricalColumns:
    # Category Indexing with StringIndexer
    stringIndexer = StringIndexer(inputCol=categoricalCol, outputCol=categoricalCol + "Index")

# Use OneHotEncoder to convert categorical variables into binary SparseVectors
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder(inputCols=[stringIndexer.getOutputCol()], outputCols=[categoricalCol + "classVec"])

# Define the pipeline based on the stages created in previous steps.
pipeline = Pipeline(stages=[stringIndexer, encoder])

# Define the pipeline model.
transform_mdl = pipeline.fit(trainDF)
trainDF21=transform_mdl.transform(trainDF)
trainDF21.show()

+---+------+--------+---------+-----------+---------+---------+-----------+--------------+
|age|   bmi|children|   region|    charges|gender_cd|smoker_cd|regionIndex|regionclassVec|
+---+------+--------+---------+-----------+---------+---------+-----------+--------------+
| 18| 15.96|       0|northeast|  1694.7964|        0|        0|        3.0|     (3,[],[])|
| 18| 17.29|       2|northeast| 12829.4551|        0|        1|        3.0|     (3,[],[])|
| 18| 21.47|       0|northeast|  1702.4553|        0|        0|        3.0|     (3,[],[])|
| 18|21.565|       0|northeast|13747.87235|        0|        1|        3.0|     (3,[],[])|
| 18| 21.66|       0|northeast| 14283.4594|        1|        1|        3.0|     (3,[],[])|
| 18| 22.99|       0|northeast|  1704.5681|        0|        0|        3.0|     (3,[],[])|
| 18|23.085|       0|northeast| 1704.70015|        0|        0|        3.0|     (3,[],[])|
| 18| 23.32|       1|southeast|  1711.0268|        0|        0|        0.0| (3,[0],[1.0])|

### Combine all feature columns into a single feature vector

Most MLlib algorithms require a single features column as input. Each row in this column contains a vector of data points corresponding to the set of features used for prediction.

MLlib provides the `VectorAssembler` transformer to create a single vector column from a list of columns.

The following code block illustrates how to use VectorAssembler.

For more information: [VectorAssembler](https://spark.apache.org/docs/latest/ml-features.html#vectorassembler)

In [18]:
# Linear regression expect a vector input
vecAssembler = VectorAssembler(inputCols=['age','bmi','children','smoker_cd','gender_cd','regionclassVec'], outputCol="features")
vecTrainDF = vecAssembler.transform(trainDF21)

# Take a look at the data
display(vecTrainDF)

DataFrame[age: int, bmi: double, children: int, region: string, charges: double, gender_cd: int, smoker_cd: int, regionIndex: double, regionclassVec: vector, features: vector]

In [19]:
display(vecTrainDF)

DataFrame[age: int, bmi: double, children: int, region: string, charges: double, gender_cd: int, smoker_cd: int, regionIndex: double, regionclassVec: vector, features: vector]

In [20]:
# Create linear regression
lr = LinearRegression(featuresCol="features", labelCol="charges")
# Fit the linear regresssion model
lrModel = lr.fit(vecTrainDF)
predict_train = lrModel.transform(vecTrainDF)

In [21]:
# Make predictions on testing dataset
testDF21=transform_mdl.transform(testDF) #do the data transformation using saved parameters from training
vecTestDF = vecAssembler.transform(testDF21) #do the feature transformation using vector assembler
predict_test = lrModel.transform(vecTestDF) #make predictions using the trained model

# Take a look at the output
display(predict_test.select("features", "charges", "prediction"))

DataFrame[features: vector, charges: double, prediction: double]

In [22]:
# Create regression evaluator
regressionEvaluator = RegressionEvaluator(predictionCol="prediction", labelCol="charges", metricName="rmse")
# RMSE
rmse = regressionEvaluator.evaluate(predict_test)
print(f"The Test RMSE for the linear regression model is {rmse:0.2f}")
rmsetr = regressionEvaluator.evaluate(predict_train)
print(f"The Train RMSE for the linear regression model is {rmsetr:0.2f}")

# MSE
mse = regressionEvaluator.setMetricName("mse").evaluate(predict_test)
print(f"The Test MSE for the linear regression model is {mse:0.2f}")
msetr = regressionEvaluator.setMetricName("mse").evaluate(predict_train)
print(f"The Train MSE for the linear regression model is {msetr:0.2f}")

# R2
r2 = regressionEvaluator.setMetricName("r2").evaluate(predict_test)
print(f"The Test R2 for the linear regression model is {r2:0.2f}")
trainr2 = regressionEvaluator.setMetricName("r2").evaluate(predict_train)
print(f"The Train R2 for the linear regression model is {trainr2:0.2f}")

# MAE
mae = regressionEvaluator.setMetricName("mae").evaluate(predict_test)
print(f"The Test MAE for the linear regression model is {mae:0.2f}")
maetr = regressionEvaluator.setMetricName("mae").evaluate(predict_train)
print(f"The Train MAE for the linear regression model is {maetr:0.2f}")

# Visualize the data
#display(predict_test.select("charges", "prediction"))

The Test RMSE for the linear regression model is 6541.34
The Train RMSE for the linear regression model is 5790.99
The Test MSE for the linear regression model is 42789184.28
The Train MSE for the linear regression model is 33535594.32
The Test R2 for the linear regression model is 0.70
The Train R2 for the linear regression model is 0.77
The Test MAE for the linear regression model is 4368.83
The Train MAE for the linear regression model is 3956.60


In [23]:
print("Coefficients: \n" + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

Coefficients: 
[266.14327319957306,305.58895187335605,657.9542238790278,24505.841413050708,-6.143081589793528,-790.7327504859998,405.4512999797808,-563.7329390258529]
Intercept: -12013.99781240657


## Assignment 1.1

1. Create a linear Regression model using the 50/50 train/test split
1. Rebuild the model by dropping the smoker variable
1. How does the RMSE and MAE values change by dropping the Smoker variable